In [ ]:
import os
from pprint import pprint
import numpy as np
import torch
from torch.utils.data import DataLoader


from src.utils import load_json_data, read_config_file
from src.data import Sentences_Dataset
from src.model import LSTM_Trainer
from src.plot import plot_training_results

# Read Training Results

In [ ]:
plot_training_results(res_file="file:./ckpt/training_metrics.csv", type_res="loss", r_mode="epoch")
plot_training_results(res_file="file:./ckpt/training_metrics.csv", type_res="acc", r_mode="epoch")
plot_training_results(res_file="file:./ckpt/training_metrics.csv", type_res="loss", r_mode="step")
plot_training_results(res_file="file:./ckpt/training_metrics.csv", type_res="acc", r_mode="step")

# Reload the Model for Evaluation and Test

In [ ]:
# Read configuration file
cfg = read_config_file("configs/config.yaml")
print("== Configuration File ==")
pprint(cfg)

# Setup the device for training
device = torch.device(cfg["project"]["device"])
print(f"Device used: {device}")

# Prepare Folders and Paths
root_path = "../"
cwd = os.getcwd()
data_path = os.path.join(root_path, cfg["paths"]["data"])
    # Dataset Paths
valid_set_path = os.path.join(data_path, "dev.jsonl")
test_set_path = os.path.join(data_path, "test.jsonl")
    # Utilities/Hyper-Parameters
utilities_path = os.path.join(cwd, cfg["paths"]["utils_path"])
    # Vocabularies
vocabulary_path = os.path.join(cwd, cfg["paths"]["dict_path"])

HYPERPARMS_ID = cfg["project"]["hp"]["id"]
hyperparams = torch.load(os.path.join(utilities_path, f"hyper_params_{HYPERPARMS_ID}.params"))["hyperparams"]
pprint(hyperparams)
print(f"-- Hyper-Parameters Correctly Loaded from hyper_params_{HYPERPARMS_ID}.params --")
ID_VOCAB = cfg["project"]["vocab"]["id"]
vocabulary_name = f"vocabulary_{ID_VOCAB}.vocab"
VOCABULARIES = torch.load(os.path.join(vocabulary_path, vocabulary_name))["vocabularies"]
print(f"-- Vocabularies Correctly Loaded from {vocabulary_name} --")

model_trainer = LSTM_Trainer(hyperparams=hyperparams,
                             dictionaries=VOCABULARIES,
                             load_checkpoint=True,
                             embed_state=None,
                             labels_weigths=None,
                             dropout=False)

## Model Evaluation (dev.jsonl)

In [ ]:
randomize_eval_data = False
json_data_evaluation = load_json_data(valid_set_path)
evaluation_data = Sentences_Dataset(raw_data=json_data_evaluation,
                                    params=hyperparams, is_training=False)

evaluation_data.encode_data(word2id=VOCABULARIES['word2idx'], label2id=VOCABULARIES["label2idx"],
                            max_length=evaluation_data.max_samp_length)
eval_data_length = len(evaluation_data.encoded_data)
print("  - Evaluation Dataset Length: ", eval_data_length)
    # Add UNK Randomness
if randomize_eval_data == True:
    n_random = int(0.33*len(evaluation_data.encoded_data))
    print("N° of senteces to be randomized: ", n_random)
    evaluation_data.randomize_encoded_n_grams(unk_token=VOCABULARIES["word2idx"]["<unk>"],
                                              n=n_random)
evaluation_dataset = DataLoader(evaluation_data.encoded_data,
                                batch_size=hyperparams["batch_size"], shuffle=False)
print("  - DataLoader is now ready for the evaluation")
print("-- Model Evaluation --")
model_trainer.evaluate(hyperparams, evaluation_dataset)

## Model Testing (test.jsonl)

In [ ]:
randomize_test_data = True
json_data_test = load_json_data(test_set_path)
test_data = Sentences_Dataset(raw_data=json_data_test, params=hyperparams)

test_data.encode_data(word2id=VOCABULARIES['word2idx'], label2id=VOCABULARIES["label2idx"],
                      max_length=test_data.max_samp_length)
test_data_length = len(test_data.encoded_data)
print("  - Test Dataset Length: ", test_data_length)
    # Add UNK Randomness
if randomize_test_data == True:
    n_random = int(0.33*len(test_data.encoded_data))
    print("N° of senteces to be randomized: ", n_random)
    test_data.randomize_encoded_n_grams(unk_token=VOCABULARIES["word2idx"]["<unk>"],
                                        n=n_random)
test_dataset = DataLoader(test_data.encoded_data,
                          batch_size=hyperparams["batch_size"], shuffle=False)
print("  - DataLoader is now ready for the test")
print("-- Model Test --")
model_trainer.evaluate(hyperparams, test_dataset)